# Fine-tune Llama 3.2 1B on Smart MCQ (Colab-friendly)

**What this notebook does**
1. Loads **`unsloth/Llama-3.2-1B`** from Hugging Face (public re-upload → **no `hf_token` / Meta gate**)
2. Uses plain **Transformers + PEFT (LoRA) + PyTorch** (no Unsloth library)
3. Fine-tunes for 5-way multiple choice (same idea as a DistilGPT-2 MCQ classifier)
4. Predicts **top-3 answers** for a Kaggle-style submission

**How the MCQ model works (simple)**
- For each question, build 5 texts: `Question: ...\nAnswer: <option text>`
- Run the LLM on each option
- Take the **last real token** hidden state → a linear score head
- Softmax / argmax over the 5 scores → best letter(s)

**Colab setup**
- Runtime → Change runtime type → **T4 GPU**
- Upload `train.csv` (and optional `test.csv`) from the competition folder


## 0. Install packages
Restart the runtime only if Colab asks you to after installing.


In [ ]:
# Core stack for free Colab
!pip -q install -U transformers peft accelerate bitsandbytes datasets pandas tqdm sentencepiece

# Optional: quieter logs
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_XET"] = "1"  # more reliable downloads on some networks


## 1. Upload data

Upload **`train.csv`** (required) and **`test.csv`** (optional, for submission).

Expected columns: `id`, `prompt`, `A`, `B`, `C`, `D`, `E`, and for train also `answer`.


In [ ]:
from google.colab import files
import os

# Skip this cell if you already put train.csv / test.csv in the working directory
uploaded = files.upload()  # choose train.csv and optionally test.csv
print("Files here:", [f for f in os.listdir(".") if f.endswith(".csv")])


## 2. Imports & config


In [ ]:
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from tqdm.auto import tqdm

from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from peft import LoraConfig, get_peft_model, TaskType

# -------------------- knobs you can change --------------------
MODEL_NAME = "unsloth/Llama-3.2-1B"  # public; no HF login needed
MAX_LENGTH = 256                     # raise if options are long (uses more VRAM)
BATCH_SIZE = 1                       # each sample = 5 sequences; keep small on free T4
NUM_EPOCHS = 2
LEARNING_RATE = 2e-4                 # LoRA can take a higher LR than full FT
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.05
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
VAL_FRACTION = 0.15
SEED = 42
EVAL_EVERY = 50                      # steps between quick loss prints
# --------------------------------------------------------------

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))


## 3. Dataset

For **each** option letter we build one string:

```
Question: <prompt>
Answer: <option text>
```

The model scores all five; the correct letter is the training label.


In [ ]:
OPTIONS = ["A", "B", "C", "D", "E"]
LABEL_MAP = {k: i for i, k in enumerate(OPTIONS)}


class MCQDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=MAX_LENGTH, is_test=False):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        prompt = str(row["prompt"])

        texts = [
            f"Question: {prompt}\nAnswer: {str(row[opt])}"
            for opt in OPTIONS
        ]

        # Tokenize each option separately (no padding yet)
        encoded_list = []
        for text in texts:
            enc = self.tokenizer(
                text,
                add_special_tokens=True,
                truncation=True,
                max_length=self.max_length,
                return_attention_mask=False,
            )
            ids = enc["input_ids"]
            # Keep the END of long texts (answer lives at the end)
            if len(ids) > self.max_length:
                ids = ids[-self.max_length:]
            encoded_list.append(torch.tensor(ids, dtype=torch.long))

        label = -1 if self.is_test else LABEL_MAP[str(row["answer"]).strip()]
        q_id = int(row["id"]) if "id" in row else idx
        return encoded_list, label, q_id


def mcq_collate_fn(batch):
    """Left-pad so the last position is always a real token (good for last-token pooling)."""
    all_ids = [item[0] for item in batch]
    labels = [item[1] for item in batch]
    q_ids = [item[2] for item in batch]

    pad_id = tokenizer.pad_token_id
    max_len = max(len(seq) for seq_list in all_ids for seq in seq_list)
    bsz = len(batch)

    input_ids = torch.full((bsz, 5, max_len), pad_id, dtype=torch.long)
    attention_mask = torch.zeros((bsz, 5, max_len), dtype=torch.long)

    for b, seq_list in enumerate(all_ids):
        for o, seq in enumerate(seq_list):
            L = len(seq)
            input_ids[b, o, -L:] = seq
            attention_mask[b, o, -L:] = 1

    return input_ids, attention_mask, torch.tensor(labels, dtype=torch.long), q_ids


## 4. Model: Llama backbone + tiny score head

We use `AutoModel` (encoder-style hidden states, not generation).

`score_head`: linear layer maps last-token embedding → 1 score per option.


In [ ]:
class LlamaForMultipleChoice(nn.Module):
    def __init__(self, base_model, hidden_size):
        super().__init__()
        self.base_model = base_model
        self.score_head = nn.Linear(hidden_size, 1, bias=False)

    def forward(self, input_ids, attention_mask):
        # input_ids: (batch, 5, seq)
        bsz, n_choices, seq_len = input_ids.shape
        flat_ids = input_ids.view(-1, seq_len)
        flat_mask = attention_mask.view(-1, seq_len)

        outputs = self.base_model(input_ids=flat_ids, attention_mask=flat_mask)
        hidden = outputs.last_hidden_state  # (bsz*5, seq, hidden)

        # With left padding, index -1 is the last real token
        last = hidden[:, -1, :].float()    # (bsz*5, hidden) — fp32 for head
        scores = self.score_head(last)     # (bsz*5, 1)
        return scores.view(bsz, n_choices) # (bsz, 5)


## 5. Load tokenizer + model + LoRA

**Why LoRA?** Full fine-tuning 1B params + 5 options per step is tight on free Colab. LoRA trains a few small adapter matrices instead.


In [ ]:
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

# Llama often has no pad token; reuse EOS for padding
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

print("Loading base model (this downloads ~2GB once)...")
# float16 on GPU; float32 on CPU fallback
dtype = torch.float16 if device.type == "cuda" else torch.float32

base = AutoModel.from_pretrained(
    MODEL_NAME,
    torch_dtype=dtype,
    low_cpu_mem_usage=True,
)

# Attach LoRA adapters to attention projections
lora_config = LoraConfig(
    task_type=TaskType.FEATURE_EXTRACTION,  # we only need hidden states
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

base = get_peft_model(base, lora_config)
base.print_trainable_parameters()

hidden_size = base.config.hidden_size
model = LlamaForMultipleChoice(base, hidden_size=hidden_size).to(device)

# Train score head + LoRA; freeze is already handled by PEFT for the backbone
# Keep the tiny head in float32 for stable CE loss (base can stay fp16)
model.score_head = model.score_head.float()
for p in model.score_head.parameters():
    p.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")


## 6. Build train / val loaders


In [ ]:
assert os.path.exists("train.csv"), "Upload train.csv first (cell above)."

full_df = pd.read_csv("train.csv")
print("Train rows:", len(full_df))
print(full_df["answer"].value_counts().sort_index())

full_ds = MCQDataset(full_df, tokenizer, max_length=MAX_LENGTH, is_test=False)
val_size = max(1, int(len(full_ds) * VAL_FRACTION))
train_size = len(full_ds) - val_size
train_ds, val_ds = random_split(
    full_ds,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED),
)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=mcq_collate_fn,
    num_workers=0,
)
val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=mcq_collate_fn,
    num_workers=0,
)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")


## 7. Train


In [ ]:
def batch_loss(input_ids, attention_mask, labels, model, device):
    input_ids = input_ids.to(device)
    attention_mask = attention_mask.to(device)
    labels = labels.to(device)
    # Cast score head inputs: base may be fp16; CE wants float logits
    logits = model(input_ids, attention_mask).float()
    return torch.nn.functional.cross_entropy(logits, labels)


@torch.no_grad()
def eval_loss_acc(loader, model, device, max_batches=None):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for i, (ids, mask, labels, _) in enumerate(loader):
        if max_batches is not None and i >= max_batches:
            break
        ids, mask, labels = ids.to(device), mask.to(device), labels.to(device)
        logits = model(ids, mask).float()
        loss = torch.nn.functional.cross_entropy(logits, labels, reduction="sum")
        total_loss += loss.item()
        pred = logits.argmax(dim=-1)
        correct += (pred == labels).sum().item()
        total += labels.numel()
    model.train()
    return total_loss / max(total, 1), correct / max(total, 1)


# Optimizer: only parameters that require grad
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

num_training_steps = NUM_EPOCHS * len(train_loader)
num_warmup = int(WARMUP_RATIO * num_training_steps)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup,
    num_training_steps=num_training_steps,
)

print(f"Steps/epoch={len(train_loader)}, total_steps={num_training_steps}, warmup={num_warmup}")
print("--- training ---")

global_step = 0
history = []

for epoch in range(NUM_EPOCHS):
    model.train()
    running = 0.0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")

    for input_ids, attention_mask, labels, _ in pbar:
        optimizer.zero_grad(set_to_none=True)
        loss = batch_loss(input_ids, attention_mask, labels, model, device)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        running += loss.item()
        global_step += 1
        pbar.set_postfix(loss=f"{loss.item():.3f}")

        if global_step % EVAL_EVERY == 0:
            tr_loss, tr_acc = eval_loss_acc(train_loader, model, device, max_batches=10)
            va_loss, va_acc = eval_loss_acc(val_loader, model, device, max_batches=10)
            history.append({"step": global_step, "train_loss": tr_loss, "val_loss": va_loss,
                            "train_acc": tr_acc, "val_acc": va_acc})
            print(
                f"step {global_step:04d} | "
                f"train loss {tr_loss:.3f} acc {tr_acc*100:.1f}% | "
                f"val loss {va_loss:.3f} acc {va_acc*100:.1f}%"
            )

    # Full-ish epoch metrics (may take a minute)
    tr_loss, tr_acc = eval_loss_acc(train_loader, model, device)
    va_loss, va_acc = eval_loss_acc(val_loader, model, device)
    print(
        f"Epoch {epoch+1} done | "
        f"Train loss {tr_loss:.3f} acc {tr_acc*100:.2f}% | "
        f"Val loss {va_loss:.3f} acc {va_acc*100:.2f}%"
    )

print("Training finished.")


## 8. Quick sanity check on a few val examples


In [ ]:
model.eval()
id_to_letter = np.array(OPTIONS)

shown = 0
with torch.no_grad():
    for input_ids, attention_mask, labels, q_ids in val_loader:
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        logits = model(input_ids, attention_mask).float().cpu().numpy()
        labels = labels.numpy()

        for i in range(len(q_ids)):
            top3 = id_to_letter[np.argsort(logits[i])[::-1][:3]]
            gold = id_to_letter[labels[i]]
            print(f"id={q_ids[i]}  gold={gold}  top3={' '.join(top3)}  scores={np.round(logits[i], 3)}")
            shown += 1
            if shown >= 5:
                break
        if shown >= 5:
            break


## 9. Predict on `test.csv` → submission file

Competition format: top-3 letters space-separated, e.g. `B A C`.


In [ ]:
SUBMISSION_PATH = "submission_llama1b_mcq.csv"

if not os.path.exists("test.csv"):
    print("No test.csv found — skipping submission. Upload it and re-run this cell.")
else:
    test_df = pd.read_csv("test.csv")
    test_ds = MCQDataset(test_df, tokenizer, max_length=MAX_LENGTH, is_test=True)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=mcq_collate_fn)

    model.eval()
    rows = []
    with torch.no_grad():
        for input_ids, attention_mask, _, q_ids in tqdm(test_loader, desc="Predict"):
            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            logits = model(input_ids, attention_mask).float().cpu().numpy()
            for i, qid in enumerate(q_ids):
                top3 = id_to_letter[np.argsort(logits[i])[::-1][:3]]
                rows.append({"id": qid, "prediction": " ".join(top3)})

    sub = pd.DataFrame(rows)
    # Match common Kaggle column names if sample_submission differs
    if os.path.exists("sample_submission.csv"):
        sample = pd.read_csv("sample_submission.csv")
        print("sample_submission columns:", list(sample.columns))
        # rename to match sample if needed
        if list(sample.columns) == ["ID", "Prediction"]:
            sub = sub.rename(columns={"id": "ID", "prediction": "Prediction"})
        elif list(sample.columns) == ["id", "prediction"]:
            pass
        else:
            # best-effort: keep first two cols of sample names
            cols = list(sample.columns)[:2]
            sub.columns = cols

    sub.to_csv(SUBMISSION_PATH, index=False)
    print(sub.head())
    print("Saved:", SUBMISSION_PATH)
    try:
        files.download(SUBMISSION_PATH)
    except Exception:
        pass


## 10. Save fine-tuned weights (LoRA + score head)


In [ ]:
import os
from pathlib import Path

save_dir = Path("llama1b_mcq_lora")
save_dir.mkdir(exist_ok=True)

# PEFT adapters
model.base_model.save_pretrained(save_dir / "lora_adapter")
tokenizer.save_pretrained(save_dir / "lora_adapter")

# Classification head
torch.save(model.score_head.state_dict(), save_dir / "score_head.pt")

print("Saved to", save_dir.resolve())
print("Contents:", list(save_dir.rglob("*")))


## How to reload later

```python
from peft import PeftModel
from transformers import AutoModel, AutoTokenizer

tok = AutoTokenizer.from_pretrained("llama1b_mcq_lora/lora_adapter")
base = AutoModel.from_pretrained("unsloth/Llama-3.2-1B", torch_dtype=torch.float16)
base = PeftModel.from_pretrained(base, "llama1b_mcq_lora/lora_adapter")
model = LlamaForMultipleChoice(base, hidden_size=base.config.hidden_size)
model.score_head.load_state_dict(torch.load("llama1b_mcq_lora/score_head.pt"))
model.to("cuda").eval()
```

### Tips if you run out of GPU memory
- Lower `MAX_LENGTH` (e.g. 192 or 128)
- Keep `BATCH_SIZE = 1`
- Reduce `LORA_R` (e.g. 8)
- Train on a subset: `full_df = full_df.sample(800, random_state=42)`

### Tips for a better score
- Train more epochs (3–5) if val accuracy still rising
- Try `unsloth/Llama-3.2-1B-Instruct` (also public) — often better at following Q/A format
- Balance rare answer letters (your `solution.ipynb` had a balancing helper)
